# Day 043 Solution — Natural Language → SQL

get_db_schema, build_sql_prompt, extract_sql, is_safe_sql, ask_db. All data and functions defined inline. Uses in-memory SQLite + Ollama llama3.2.

In [ ]:
import warnings
warnings.filterwarnings('ignore')


import sqlite3

def setup_db(conn):
    cur = conn.cursor()
    cur.execute('''
        CREATE TABLE IF NOT EXISTS orders (
            order_id  INTEGER PRIMARY KEY,
            product   TEXT,
            category  TEXT,
            region    TEXT,
            price     REAL,
            quantity  INTEGER,
            revenue   REAL
        )''')
    cur.execute('''
        CREATE TABLE IF NOT EXISTS products (
            product    TEXT PRIMARY KEY,
            category   TEXT,
            unit_price REAL
        )''')
    rows = [
        (1,'Widget','Electronics','North',25.0,10,250.0),
        (2,'Gadget','Electronics','South',150.0,3,450.0),
        (3,'Widget','Electronics','South',25.0,5,125.0),
        (4,'Doohickey','Accessories','East',8.0,50,400.0),
        (5,'Gadget','Electronics','East',150.0,7,1050.0),
        (6,'Widget','Electronics','East',25.0,4,100.0),
        (7,'Doohickey','Accessories','North',8.0,20,160.0),
        (8,'Gadget','Electronics','North',150.0,2,300.0),
        (9,'Widget','Electronics','West',25.0,6,150.0),
        (10,'Doohickey','Accessories','South',8.0,15,120.0),
        (11,'Thingamajig','Accessories','North',200.0,1,200.0),
        (12,'Thingamajig','Accessories','East',200.0,4,800.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO orders VALUES (?,?,?,?,?,?,?)', rows
    )
    products = [
        ('Widget','Electronics',25.0),
        ('Gadget','Electronics',150.0),
        ('Doohickey','Accessories',8.0),
        ('Thingamajig','Accessories',200.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO products VALUES (?,?,?)', products
    )
    conn.commit()


def run_query(conn, sql, params=()):
    cur = conn.cursor()
    cur.execute(sql, params)
    cols = [col[0] for col in cur.description]
    return [dict(zip(cols, row)) for row in cur.fetchall()]


def get_db_schema(conn) -> str:
    cur = conn.cursor()
    cur.execute(
        "SELECT name, sql FROM sqlite_master WHERE type='table' ORDER BY name"
    )
    rows = cur.fetchall()
    if not rows:
        return 'No tables found.'
    parts = []
    for name, ddl in rows:
        parts.append(f'Table: {name}')
        parts.append(ddl)
        parts.append('')
    return '\n'.join(parts).strip()


def build_sql_prompt(question: str, schema_str: str) -> str:
    return (
        'You are a SQL expert. Write a SQLite SELECT query to answer the question.\n\n'
        'Requirements:\n'
        '- Use only SELECT statements.\n'
        '- The database schema is provided below.\n'
        '- Respond with ONLY a fenced SQL code block, no explanation.\n\n'
        f'Schema:\n{schema_str}\n\n'
        f'Question: {question}'
    )


import re

def extract_sql(response: str) -> str:
    fence = '`' * 3
    match = re.search(fence + r'sql\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    match = re.search(fence + r'\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()


import re

def is_safe_sql(sql: str) -> bool:
    normalized = re.sub(r'--[^\n]*', '', sql)
    normalized = re.sub(r'/\*.*?\*/', '', normalized, flags=re.DOTALL)
    normalized = normalized.strip().lower()
    if not normalized.startswith('select'):
        return False
    if ';' in normalized:
        return False
    return True


import ollama

def ask_db(conn, question: str, model: str = 'llama3.2') -> str:
    schema = get_db_schema(conn)
    prompt = build_sql_prompt(question, schema)
    resp   = ollama.chat(model=model,
                         messages=[{'role': 'user', 'content': prompt}])
    sql    = extract_sql(resp['message']['content'])
    if not is_safe_sql(sql):
        return f'Unsafe SQL rejected: {sql[:120]}'
    try:
        rows = run_query(conn, sql)
    except Exception as e:
        return f'Query error: {e}'
    if not rows:
        return 'No results found.'
    return str(rows)

## Step 1 — Create and Populate Database

In [ ]:
conn = sqlite3.connect(':memory:')
setup_db(conn)

n = conn.execute('SELECT COUNT(*) FROM orders').fetchone()[0]
assert n == 12
print(f'orders: {n} rows')

## Step 2 — get_db_schema

In [ ]:
schema = get_db_schema(conn)
assert isinstance(schema, str) and len(schema) > 0
assert 'orders' in schema.lower()
assert 'products' in schema.lower()
assert 'revenue' in schema.lower()
print(schema)

## Step 3 — build_sql_prompt

In [ ]:
q = 'How many orders are there?'
prompt = build_sql_prompt(q, schema)
assert isinstance(prompt, str)
assert q in prompt
assert schema in prompt
print(f'Prompt length: {len(prompt)} chars')

## Step 4 — extract_sql

In [ ]:
fence = '`' * 3
sql_text = 'SELECT COUNT(*) FROM orders'
response = fence + 'sql\n' + sql_text + '\n' + fence
extracted = extract_sql(response)
assert extracted == sql_text, f'expected {sql_text!r}, got {extracted!r}'
print(f'Extracted: {extracted}')

## Step 5 — is_safe_sql

In [ ]:
assert is_safe_sql('SELECT * FROM orders') is True
assert is_safe_sql("INSERT INTO orders VALUES (99,'x','y','z',0,0,0)") is False
assert is_safe_sql('DROP TABLE orders') is False
assert is_safe_sql('SELECT * FROM orders; DROP TABLE orders') is False
print('is_safe_sql: all checks passed')

## Step 6 — ask_db (full pipeline, uses Ollama)

In [ ]:
result = ask_db(conn, 'How many orders are there?')
assert isinstance(result, str)
assert len(result) > 0
print(f'ask_db result: {result}')

result2 = ask_db(conn, 'What is the total revenue from all orders?')
assert isinstance(result2, str)
assert len(result2) > 0
print(f'ask_db result2: {result2}')

print('\nAll solution checks passed.')
conn.close()